# Agreement-balanced transcript dataset generation

This notebook generates noisy-channel transcript rows with uniform coverage of the full Cartesian agreement-pattern grid for fixed candidates $X$ and $Y$. For $K$ questions, each candidate has $2^K$ possible agreement vectors, giving $4^K$ grid cells. `NUM_QUESTION_SETS` is the number of independently sampled compatible question schedules **per grid cell**.

The resulting `TranscriptDataset` uses the same prompt, posterior, row, `dataset.jsonl`, and manifest formats as the ordinary transcript generator. Agreement-specific grid coordinates are included on every row.

This balances exact agreement **vectors**. If rows are later collapsed to total agreement counts, the usual combinatorial multiplicities remain (for $K=3$: 1, 3, 3, 1 patterns at totals 0, 1, 2, 3).

In [ ]:
from __future__ import annotations

import sys
from collections import Counter
from pathlib import Path

from IPython.display import display
from transformers import AutoProcessor

REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").exists()
)
sys.path.insert(0, str(REPO_ROOT))

from mats_experiments.noisy_channel_bayesian import (
    AgreementSubsetQuestion,
    AgreementTranscriptDatasetGenerator,
    NoisyChannelBayesianEnvironment,
    SystemPrompt,
    TokenizerBinding,
    TranscriptDataset,
    XVsYPosteriorProbe,
)

## Configuration

The default creates two schedules per agreement cell. Increase `NUM_QUESTION_SETS` for more independently sampled realizations of each cell.

In [ ]:
N = 8
K = 3
X = 2
Y = 7
SUBSET_SIZE = 4
NUM_QUESTION_SETS = 2  # Per (X agreement pattern, Y agreement pattern) cell.
SEED = 20260908

R_VALUES = (0.1, 0.5, 0.9)
REASONING_VALUES = (False, True)
CONTROL_POSITIONAL_BIAS = True
SYSTEM_PROMPT_TEXT = (
    "You are a Bayesian reasoner. Follow the user's game rules exactly. "
    "Use plaintext only."
)

TOKENIZER_NAME_OR_PATH = REPO_ROOT / "models" / "Qwen--Qwen3.5-4B"
OUTPUT_DIR = REPO_ROOT / "artifacts" / "noisy_channel_agreement_grid_dataset"

AGREEMENT_PATTERNS_PER_CANDIDATE = 2**K
AGREEMENT_CELLS = 4**K
PRESENTATIONS = 2 if CONTROL_POSITIONAL_BIAS else 1
EXPECTED_ROWS = (
    len(R_VALUES)
    * len(REASONING_VALUES)
    * AGREEMENT_CELLS
    * NUM_QUESTION_SETS
    * PRESENTATIONS
)
print({
    "agreement_grid": (AGREEMENT_PATTERNS_PER_CANDIDATE,) * 2,
    "agreement_cells": AGREEMENT_CELLS,
    "question_sets_per_cell": NUM_QUESTION_SETS,
    "expected_rows": EXPECTED_ROWS,
    "output_dir": str(OUTPUT_DIR),
})

## Bind the tokenizer

Dataset rows contain the exact serialized prompt and input token IDs, so generation is bound to a tokenizer just like `TranscriptDatasetGenerator`.

In [ ]:
processor = AutoProcessor.from_pretrained(
    str(TOKENIZER_NAME_OR_PATH),
    trust_remote_code=False,
    local_files_only=True,
)
tokenizer = getattr(processor, "tokenizer", processor)
tokenizer_binding = TokenizerBinding(tokenizer, enable_thinking=False)

## Generate and save

For every grid cell, `AgreementSubsetQuestion` samples observed reports and fixed-size membership subsets that realize the requested agreement vectors. The same sampled schedules are reused across reliability and reasoning conditions.

In [ ]:
environments = tuple(
    NoisyChannelBayesianEnvironment(
        n=N,
        k=K,
        r_values=reliability,
        control_positional_bias=CONTROL_POSITIONAL_BIAS,
    )
    for reliability in R_VALUES
)
probes = tuple(
    XVsYPosteriorProbe(
        x=X,
        y=Y,
        reasoning=reasoning,
        allow_same=False,
        call_layout="conversation",
    )
    for reasoning in REASONING_VALUES
)
question = AgreementSubsetQuestion(subset_size=SUBSET_SIZE, sort=True)

dataset = AgreementTranscriptDatasetGenerator(
    environment=environments,
    question=question,
    probe=probes,
    tokenizer_binding=tokenizer_binding,
    system_prompt=SystemPrompt(SYSTEM_PROMPT_TEXT),
    seed=SEED,
).generate(num_question_sets=NUM_QUESTION_SETS)
dataset_path = dataset.save(OUTPUT_DIR)
print(f"Saved {len(dataset):,} rows to {dataset_path}")

## Audit uniform grid coverage

Each reliability, reasoning, and presentation slice must contain every agreement cell exactly `NUM_QUESTION_SETS` times. The computed canonical agreement vectors must also equal the requested targets on every row.

In [ ]:
assert len(dataset) == EXPECTED_ROWS
assert dataset.manifest["agreement_cell_count"] == AGREEMENT_CELLS
assert dataset.manifest["num_question_sets_per_agreement_cell"] == NUM_QUESTION_SETS
assert dataset.manifest["num_question_sets"] == AGREEMENT_CELLS * NUM_QUESTION_SETS

coverage = Counter(
    (
        row["environment_parameter_index"],
        row["probe_parameter_index"],
        row["presentation_order"],
        row["agreement_x_pattern"],
        row["agreement_y_pattern"],
    )
    for row in dataset
)
assert len(coverage) == len(R_VALUES) * len(REASONING_VALUES) * PRESENTATIONS * AGREEMENT_CELLS
assert set(coverage.values()) == {NUM_QUESTION_SETS}
assert all(
    row["agreement_candidate_1_by_question"]
    == row["target_agreement_x_by_question"]
    and row["agreement_candidate_2_by_question"]
    == row["target_agreement_y_by_question"]
    for row in dataset
)
print("Uniform agreement-grid audit passed.")

In [ ]:
patterns = dataset.manifest["agreement_patterns"]
first_slice = [
    row for row in dataset
    if row["environment_parameter_index"] == 0
    and row["probe_parameter_index"] == 0
    and row["presentation_order"] == "C1_C2"
]
first_slice_counts = Counter(
    (row["agreement_x_pattern"], row["agreement_y_pattern"])
    for row in first_slice
)
print("      Y patterns →", *patterns)
for x_pattern in patterns:
    print(
        f"X={x_pattern}",
        *[first_slice_counts[(x_pattern, y_pattern)] for y_pattern in patterns],
    )

## Inspect compatible schedules for one cell

In [ ]:
examples = [
    {
        "question_set_index": row["question_set_index"],
        "agreement_question_set_index": row["agreement_question_set_index"],
        "agreement_x_pattern": row["agreement_x_pattern"],
        "agreement_y_pattern": row["agreement_y_pattern"],
        "membership_sets": row["membership_sets"],
        "observed_reports": row["observed_reports"],
    }
    for row in first_slice
    if row["agreement_x_pattern"] == "NYY"
    and row["agreement_y_pattern"] == "YNY"
]
display(examples)

## Verify the saved JSONL artifact

In [ ]:
reloaded = TranscriptDataset.load(OUTPUT_DIR)
assert [row["row_id"] for row in reloaded] == [row["row_id"] for row in dataset]
assert reloaded.manifest == dataset.manifest
print({
    "dataset_jsonl": str(OUTPUT_DIR / "dataset.jsonl"),
    "dataset_manifest": str(OUTPUT_DIR / "dataset_manifest.json"),
    "row_count": len(reloaded),
})